In [0]:
-- ============================================================
-- SEGMENTACION RFM + AFINIDAD DE MARCA
-- Fuente: tabla `compras` (Databricks SQL)
-- Ventana: 2025-03-21 a 2025-05-04
-- ============================================================
-- Enfoque:
--   Eje 1 - RFM clasico (Recency, Frequency, Monetary) con
--           quintiles NTILE(5) por cada dimension.
--   Eje 2 - Afinidad de marca via indice HHI de concentracion.
--   Salida: tabla `customer_profiles` con un perfil accionable
--           por cliente + resumen de distribucion por segmento.
-- ============================================================

-- 0) LIMPIEZA (si no se ha corrido antes)
CREATE OR REPLACE TEMP VIEW compras_clean AS
SELECT
    UPPER(REGEXP_REPLACE(BRAND, '\\s+', ''))  AS brand,
    CAST(SKU_ID AS INT)                       AS sku_id,
    CAST(UNIT_QUANTITY AS DOUBLE)             AS units,
    CAST(VOLUME AS DOUBLE)                    AS volume,
    CAST(GMV AS DOUBLE)                       AS gmv,
    TRIM(CUSTOMER)                            AS customer,
    CAST(ORDER_DATE AS DATE)                  AS order_date,
    TRIM(POC)                                 AS poc
FROM workspace.default.prueba_bavaria_compras;


-- Fecha de referencia = ultima compra observada en la base
CREATE OR REPLACE TEMP VIEW params AS
SELECT MAX(order_date) AS ref_date FROM compras_clean;


-- ============================================================
-- 1) METRICAS RFM + AUXILIARES POR CLIENTE
-- ============================================================
CREATE OR REPLACE TEMP VIEW customer_metrics AS
SELECT
    c.customer,
    -- RFM base
    DATEDIFF(p.ref_date, MAX(c.order_date))      AS recency_days,
    COUNT(*)                                     AS frequency,
    SUM(c.gmv)                                   AS monetary,
    -- Auxiliares
    COUNT(DISTINCT c.order_date)                 AS active_days,
    COUNT(DISTINCT c.brand)                      AS brand_variety,
    COUNT(DISTINCT c.sku_id)                     AS sku_variety,
    AVG(c.gmv)                                   AS avg_ticket,
    DATEDIFF(p.ref_date, MIN(c.order_date))      AS tenure_days
FROM compras_clean c CROSS JOIN params p
GROUP BY c.customer, p.ref_date;


-- ============================================================
-- 2) SCORE RFM (QUINTILES 1-5)
--    - Recency: menos dias = mejor score (score alto)
--    - Frequency / Monetary: mas = mejor score
-- ============================================================
CREATE OR REPLACE TEMP VIEW rfm_scores AS
SELECT
    customer,
    recency_days, frequency, monetary,
    active_days, brand_variety, sku_variety, avg_ticket, tenure_days,
    -- Recency invertida (menos dias = score mas alto)
    6 - NTILE(5) OVER (ORDER BY recency_days ASC)  AS r_score,
    NTILE(5) OVER (ORDER BY frequency ASC)          AS f_score,
    NTILE(5) OVER (ORDER BY monetary ASC)           AS m_score
FROM customer_metrics;


-- ============================================================
-- 3) AFINIDAD DE MARCA (HHI + tipo)
-- ============================================================
CREATE OR REPLACE TEMP VIEW brand_shares AS
SELECT
    customer, brand,
    SUM(gmv) / SUM(SUM(gmv)) OVER (PARTITION BY customer) AS share
FROM compras_clean
GROUP BY customer, brand;


CREATE OR REPLACE TEMP VIEW brand_affinity AS
SELECT
    bs.customer,
    SUM(POWER(bs.share, 2))                            AS wallet_hhi,
    COUNT(*)                                           AS brands_bought,
    MAX(bs.share)                                      AS top_brand_share,
    -- marca principal del cliente
    FIRST(CASE WHEN rn = 1 THEN brand END)             AS top_brand
FROM (
    SELECT customer, brand, share,
           ROW_NUMBER() OVER (PARTITION BY customer ORDER BY share DESC) AS rn
    FROM brand_shares
) bs
GROUP BY bs.customer;


-- ============================================================
-- 4) TABLA FINAL DE PERFILES
-- ============================================================
CREATE OR REPLACE TABLE customer_profiles AS
SELECT
    s.customer,

    -- metricas crudas
    s.recency_days, s.frequency, s.monetary,
    s.avg_ticket, s.tenure_days, s.active_days,
    s.brand_variety, s.sku_variety,

    -- scores
    s.r_score, s.f_score, s.m_score,
    CONCAT(s.r_score, s.f_score, s.m_score)            AS rfm_code,
    (s.r_score + s.f_score + s.m_score)                AS rfm_total,

    -- afinidad de marca
    ba.wallet_hhi, ba.top_brand, ba.top_brand_share, ba.brands_bought,

    -- Eje 1: segmento RFM con nombre
    CASE
        WHEN s.r_score >= 4 AND s.f_score >= 4 AND s.m_score >= 4 THEN '01_Campeones'
        WHEN s.f_score >= 4 AND s.m_score >= 3                    THEN '02_Leales'
        WHEN s.r_score >= 4 AND s.f_score <= 3 AND s.tenure_days <= 14 THEN '03_Nuevos'
        WHEN s.r_score >= 4 AND s.f_score <= 3                    THEN '04_Potenciales_leales'
        WHEN s.r_score >= 3 AND s.m_score <= 2                    THEN '05_Prometedores'
        WHEN s.r_score BETWEEN 2 AND 3 AND s.f_score >= 3         THEN '06_Necesitan_atencion'
        WHEN s.r_score <= 2 AND s.f_score >= 3                    THEN '07_En_riesgo'
        WHEN s.r_score <= 2 AND s.f_score <= 2                    THEN '08_Hibernando'
        ELSE '09_Otros'
    END AS rfm_segment,

    -- Eje 2: tipo de afinidad de marca
    CASE
        WHEN ba.wallet_hhi > 0.7  OR ba.brands_bought <= 2 THEN 'Monogamo'
        WHEN ba.wallet_hhi < 0.3  OR ba.brands_bought >= 5 THEN 'Explorador'
        ELSE 'Mixto'
    END AS brand_affinity

FROM rfm_scores s
LEFT JOIN brand_affinity ba USING (customer);


-- ============================================================
-- 5) RESULTADOS / DISTRIBUCIONES
-- ============================================================

-- 5.1 Distribucion de clientes por segmento RFM
SELECT
    rfm_segment,
    COUNT(*)                                             AS customers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)   AS pct_customers,
    ROUND(AVG(recency_days), 1)                          AS avg_recency,
    ROUND(AVG(frequency), 1)                             AS avg_frequency,
    ROUND(AVG(monetary), 2)                              AS avg_monetary,
    ROUND(SUM(monetary), 2)                              AS total_gmv,
    ROUND(SUM(monetary) * 100.0 / SUM(SUM(monetary)) OVER (), 2) AS pct_gmv
FROM customer_profiles
GROUP BY rfm_segment
ORDER BY rfm_segment;


-- 5.2 Distribucion por afinidad de marca
SELECT
    brand_affinity,
    COUNT(*)                                             AS customers,
    ROUND(AVG(wallet_hhi), 3)                            AS avg_hhi,
    ROUND(AVG(brands_bought), 2)                         AS avg_brands,
    ROUND(AVG(monetary), 2)                              AS avg_gmv
FROM customer_profiles
GROUP BY brand_affinity
ORDER BY customers DESC;


-- 5.3 Matriz cruzada RFM x Afinidad
SELECT
    rfm_segment,
    brand_affinity,
    COUNT(*)              AS customers,
    ROUND(SUM(monetary),2) AS total_gmv
FROM customer_profiles
GROUP BY rfm_segment, brand_affinity
ORDER BY rfm_segment, brand_affinity;


-- 5.4 Top marcas por segmento (cual marca domina a cada segmento)
SELECT
    rfm_segment,
    top_brand,
    COUNT(*) AS customers
FROM customer_profiles
GROUP BY rfm_segment, top_brand
QUALIFY ROW_NUMBER() OVER (PARTITION BY rfm_segment ORDER BY customers DESC) <= 3
ORDER BY rfm_segment, customers DESC;